In [1]:
using KernelAbstractions
using AMDGPU

In [2]:
# generic kernel that can be defined wherever in a module
@kernel inbounds = true function _ka_kernel(in, out)
    id = @index(Global)
    # just call the compute_expr that will dispatch to the generated function
    out[id] = compute_expr(in[id])
end

# cache inside the module
cached_expr = Dict{Type, Expr}()

# generated function that calls the specific expr from the cache
# once this has been called, the expr is not evaluated again for the same type, i.e., it can't be changed again (but will also not be recompiled)
@generated function compute_expr(x::T) where {T}
    return cached_expr[T]
end

compute_expr (generic function with 1 method)

In [3]:
# function that generates the expression and stores it in the cache, depending on the given type
function gen_func(::Type{T}) where {T}
    if T == Float32
        cached_expr[Float32] = :(x * 2)
    elseif T == Float64
        cached_expr[Float64] = :(x * 4)
    elseif T == Int32
        cached_expr[Int32] = :(x * Int32(6))
    end

    return _ka_kernel
end

gen_func (generic function with 1 method)

In [4]:
# make a function to have non-increasing world-age
function barrier(in::AbstractVector)
    # ~~magic~~
    T = eltype(in)
    k = gen_func(T)
    # our awesome kernel has been generated and written to cache
    # now immediately call the kernel without increasing world age
    
    out = similar(in)
    k(get_backend(in), 32)(in, out; ndrange=length(in))

    return out
end

barrier (generic function with 1 method)

In [5]:
# choose some backend and one of the Types [Float32, Float64, Int32]
@show in = ROCVector(rand(Float64, 128))

barrier(in)

in = ROCVector(rand(Float64, 128)) = [0.13293873483823604, 0.04843985919195637, 0.5851432590532055, 0.0026085599450507146, 0.22738712289533403, 0.6970814867710848, 0.4642987772160879, 0.2843881468622511, 0.06174408440954715, 0.74447063762157, 0.39671601998764505, 0.7458644629747496, 0.9551984782251963, 0.7565819546164951, 0.1182538375355714, 0.8361786294401837, 0.47855966016015006, 0.9941173586090429, 0.8257367434858406, 0.875752810174406, 0.08983466059930834, 0.20820507491122497, 0.24808370522234313, 0.48880974596504245, 0.1596078905233641, 0.5084565484513894, 0.6101520936707264, 0.12501505251299794, 0.6168747578377466, 0.02750338395480767, 0.45633543239970087, 0.5643104321049099, 0.4426562023321057, 0.3984224417032455, 0.615845700238958, 0.5367986805674086, 0.7860898633111216, 0.8852077479274418, 0.026474353426086616, 0.6136840350027205, 0.4106980854291562, 0.5071880148363413, 0.8657559716625383, 0.08119315987219144, 0.23825875311752764, 0.5188333308247254, 0.6196798307945655, 0.6992

128-element ROCArray{Float64, 1, AMDGPU.Runtime.Mem.HIPBuffer}:
 0.5317549393529442
 0.19375943676782548
 2.340573036212822
 0.010434239780202859
 0.9095484915813361
 2.7883259470843393
 1.8571951088643517
 1.1375525874490044
 0.2469763376381886
 2.97788255048628
 ⋮
 0.1698157306362642
 0.03587905799331903
 2.8982992181642713
 2.9263848439778495
 0.20772394671374617
 2.1311506936386726
 3.2952124174784507
 1.3328620989862805
 3.5694135946680228